一个完整的Agent至少要包含两个关键的部分：
- **模型**：是Agent的大脑，负责推理、分析，规划任务步骤
- **工具**：是Agent的手脚，负责执行任务，与外界交互

因此，定义带有工具的Agent的基本流程如下：
- 定义工具
- 初始化模型
- 初始化Agent，绑定模型和工具

# 1.自定义工具

所谓的**工具（Tool）**，本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。因此除了定义函数外，我们还需要清晰描述这个工具，让模型知道这个工具如何使用。包括下列信息：
- 工具名
- 工具的作用
- 工具需要的参数


## 1.1.基于tool描述工具
在LangChain中，定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用：


## 1.2.使用函数名和文档注释描述工具

如果不@tool装饰器没有定义工具名和作用描述，此时：
- 工具名：默认就是函数名
- 工具所需的参数：默认就是函数的参数列表
- 工具作用的描述：默认就是函数的文档注释

In [ ]:
from langchain_classic.chat_models import init_chat_model
from langchain_core.tools import tool
from openai import base_url, api_key


# 通过tool装饰器定义工具
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [ ]:
# 定义一个查询天气的tool
@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    Get current weather and optional forecast.
    Args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

## 1.3.定义Pydantic Model描述参数
如果函数的参数比较多，而且比较复杂，此时建议通过pydantic model来描述参数列表。


In [ ]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference, default is celsius."
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )


# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


工具调用方式与普通函数调用方式一致。


In [ ]:
square_root.invoke({"x": 467})

In [ ]:
get_weather.invoke({"location": "杭州", "include_forecast": True})

## 测试

In [18]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="deepseek-chat",
    tools=[square_root, get_weather],
    system_prompt="你可以使用工具回答用户问题，调用工具时尽量使用默认参数，除非用户特别指定。"
)

NameError: name 'square_root' is not defined

In [ ]:
for token, metadata in agent.stream(
        {"messages": [HumanMessage(content="杭州接下来几天天气如何?")]},
        stream_mode="messages"
):
    print(token.content, end="", flush=True)


# 2.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：
- tavily：就是一个用来做web搜索的工具

## 2.1.基本用法
它的使用步骤是这样的：
- 注册账号，创建API_KEY
- 配置环境变量: TAVILY_API_KEY
- 安装依赖：`uv add langchain-tavily`


In [12]:
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import os

load_dotenv()
qwen_api_key = os.getenv('ALIYUN')

search_tool = TavilySearch(
    max_results=5,
    topic='general',

)


In [4]:
search_tool.invoke("萝卜纸巾猫是什么梗")

{'query': '萝卜纸巾猫是什么梗',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://unifuncs.com/s/1mqwLgi5',
   'title': '萝卜纸巾猫爆火指南 - U深搜',
   'content': '核心发现：\n\n1. "萝卜纸巾猫"是2025年底爆火的网络梗，起源于抖音博主"超级无敌大开门"训练三花猫分辨萝卜和纸巾的视频\n2. 视频中猫咪通过"挨个试探+偷瞄主人表情"的方式来蒙答案，博主每次都说"真棒"鼓励\n3. 这个梗迅速走红，多地文旅部门也加入蹭流量\n4. 引发了全网模仿热潮\n\n用户说"萝卜纸巾 真棒"很可能是在引用这个梗中的经典台词，表达对这个梗的喜爱。\n\n现在我需要深度阅读几个关键页面，获取更详细的信息来给用户一个完整的解释。我选择以下几个页面进行深度阅读：\n\n1. 虎嗅网的分析文章 - 可能有更深度的内容解读\n2. 环球网的报道 - 了解爆火的全貌\n3. 澎湃新闻的报道 - 媒体视角的报道\n\n让我并行阅读这几个页面。\n\n哈哈，我现在明白了！🐱🥕 "萝卜纸巾"原来是指2025年底爆火的网络热梗"萝卜纸巾猫"，而不是一个纸巾品牌！\n\n让我深入了解一下这个梗的来龙去脉：\n\n萝卜纸巾猫梗的起源和爆火原因\n\n萝卜纸巾猫爆火现象\n\n萝卜纸巾猫走红详情\n\n用户的问题："萝卜纸巾 真棒"看起来是在用这个梗中的经典台词来表达对这个梗的喜爱。\n\n目前我已经收集到以下关键信息： [...] 目前我已经收集到的信息：\n\n1. 起源：抖音博主"超级无敌大开门"训练三花猫分辨萝卜和纸巾的视频\n2. 核心内容：猫咪通过"偷瞄主人表情+挨个试探"的方式蒙答案\n3. 经典台词：主人每次都说"真棒"来鼓励\n4. 爆火原因：门槛低、易复刻、治愈效果、年末需要简单快乐的内容\n5. 传播情况：从宠物界扩展到文旅赛道、动物园等\n6. 数据：博主一月涨粉58万，12月21日一天涨粉超26万，截至12月24日粉丝102.2万，点赞2959.4万\n\n当前深度阅读次数：5次，已经足够支撑多源验证框架。\n\n哈哈，我明白了！您是在用 "萝卜纸巾猫" 这个202

In [27]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
qwen_model=init_chat_model(
        model='qwen-max',
        model_provider='openai',
        base_url='https://maas.qianwenaiapi.com/compatible-mode/v1',
        api_key=qwen_api_key,
    )
agent = create_agent(
    model=qwen_model,
    tools=[search_tool],
    system_prompt='你是一个智能助手，你使用工具来解决用户问题',
)

In [28]:
response = agent.invoke({'messages': [HumanMessage('爆了是什么梗')]})

In [29]:
print(response['messages'][-1].content)

"爆了"是一个网络流行语，它有几种不同的含义和用法：

1. **游戏领域**：在许多角色扮演类、休闲类以及竞技类游戏中，“爆了”可以用来形容玩家在瞬间获得大量金币或奖励的情况。例如，在《英雄联盟》或《王者荣耀》等游戏中，玩家通过完成高难度任务或赢得比赛，获得远超平常的金币，就可以说是“爆了”。

2. **现实生活**：在社交平台上，“爆了”逐渐演变成一种表达方式，通常用来形容某种突如其来的好运或意外收获。例如，突然得到大额奖金、新的工作机会或是意想不到的福气，都会被戏称为“爆了”。此外，“爆了”也被用来形容在亲密关系中的一方向另一方索要金钱的行为。

3. **负面含义**：有时“爆了”还有一种讽刺用法，用来形容啃老一族、家里蹲殴打父母逼迫他们给钱的行为。这个意思源于百度贴吧的一位用户，他长期依赖父母生活，还常常打骂父母，其行为被网友用“爆了”来比喻，将压榨父母血汗钱的行为与游戏中的获取金币行为相提并论，既讽刺又批判。

4. **饭圈文化**：“A爆了”是一种在粉丝圈内流行的表达方式，其中“A”代表一种强大、帅气或者非常吸引人的气质。“A爆了”用来形容某人特别帅气或很有魅力，尤其是在描述偶像时经常使用这种说法。

根据上下文的不同，“爆了”可能指代上述任何一种情况。如果你提到的是特定情境下的“爆了”，请提供更多背景信息以便更准确地解释。
  
参考资料：
- [Bilibili - 爆！是什么梗【伪梗指南】](https://www.bilibili.com/video/BV1Bg4y1u7H1)
- [硬派科技_硬派网 - 爆金币是什么梗？爆金币梗的意思介绍](http://news.inpai.com.cn/redian/20250701/072025_286420.html)
- [百度百科 - a爆了](https://baike.baidu.com/item/a%E7%88%86%E4%BA%86/23495068)


## 2.2.优化

目前的搜索智能体存在两个问题：
- 官方默认的tavily工具过于复杂
- 结果中不包含网页数据源，可信度低

解决思路：
- 自定义tavily工具
- 结构化输出

### 自定义tavily工具

LangChain官方提供的tavily工具包含了完整的参数列表，会导致额外的流量和Token消耗。因此，对于简单的业务，我们建议大家利用tavily自定义工具。


In [31]:
tavily = TavilySearch(max_results=5, topic='general', )

from langchain.tools import tool

@tool
def web_search(query:str) -> str:
    """search the web for information"""
    return tavily.invoke(query)

### 定义结构化输出实体


In [32]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

In [33]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

In [34]:
response = agent.invoke({'messages': [HumanMessage("我跟你爆了是什么梗")]})
print(response['structured_response'])

answer='「我跟你爆了」（也常写作「跟他爆了」「MD，跟你爆了」）是网络流行语/梗，核心意思是「**跟你拼了、跟你同归于尽、自爆式拉你下水**」——一种带着暴躁、破罐破摔、鱼死网破气质的宣言。\n\n### 出处与来源\n这个词最早可以追溯到竞技类大逃杀游戏（如《Apex英雄》）里的用法。在决赛圈胜算渺茫、拿不到第一时，玩家会选择「自爆」或极端激进的方式硬刚对手来保名次，于是有了「跟他爆了」「跟他自爆」的说法。[1][2] 之后这个句式被网络传播开来，「MD，跟你爆了」「受不了了，跟他爆了」等衍生说法在B站、抖音等平台大量出现（如碧蓝航线、崩坏星穹铁道等二次元梗视频里也频繁使用）。[3][4]\n\n### 用法与含义\n现在它的意思已经从游戏扩展到日常表达，主要有几层：\n\n1. **鱼死网破 / 同归于尽**：我不活了，也要拉你一起完蛋——「你敢这样，我跟你爆了！」\n2. **破釜沉舟的极端表态**：走投无路、豁出去了的暴躁发言，「这局输了，我跟你爆了」。[1]\n3. **情绪化、玩梗式的放狠话**：网友之间调侃、互怼时用，表达「气死我了、跟你拼了」，语气凶但其实多半是玩笑和夸张。\n4. **关系亲密的人之间用粗话表达不满**：也有说法认为它结合了方言（如重庆话）与网络流行语，指熟人之间爆粗口、起外号来表达不满的现象。[5]\n\n### 梗的气质\n它带有强烈的**「无能狂怒」「破罐破摔」式的喜剧感**——越是走投无路、输得一塌糊涂，越要喊一句「我跟你爆了」，因此常被做成表情包、鬼畜视频，用来夸张地表达愤怒或绝望。用法上基本等同于「我跟你拼了 / 我不活了」，语气夸张、带点自嘲。\n\n简单说：**「跟你爆了」= 跟你拼了、同归于尽、豁出去了**，是句带着暴躁喜感的网络狠话。' reference=[Reference(title='「爆了」是什么梗？ - 知乎', url='https://www.zhihu.com/question/613068209'), Reference(title='“MD，跟你爆了”是什么梗？？？【碧蓝航线梗指南】_哔哩哔哩_bilibili', url='https://www.bilibili.com/video/av325309206'), Reference(title='请问“跟他爆了”是什么梗 - 豆瓣